In [ ]:
import xarray as xr

def get_field(data, field_name: str):
    """
    Returns DataArray(time, asset) for the requested field, handling both:
    - DataArray with 'field' dim
    - Dataset with variables
    """
    # Case A: DataArray with field dimension
    if isinstance(data, xr.DataArray) and ("field" in data.dims or "field" in data.coords):
        if field_name not in list(data.coords["field"].values):
            raise KeyError(f"Field '{field_name}' not found. Available fields: {list(data.coords['field'].values)}")
        return data.sel(field=field_name)

    # Case B: Dataset with variables
    if isinstance(data, xr.Dataset):
        if field_name in data.data_vars:
            return data[field_name]

        # Try common alternatives
        alternatives = {
            "close": ["close_adj", "adj_close", "last", "price", "Close"],
            "is_liquid": ["liquid", "isLiquid", "IsLiquid"],
        }
        for alt in alternatives.get(field_name, []):
            if alt in data.data_vars:
                return data[alt]

        raise KeyError(
            f"Variable '{field_name}' not found. Available variables: {list(data.data_vars)}"
        )

    raise TypeError(f"Unsupported data type: {type(data)}")

In [ ]:
import qnt.data as qndata
import xarray as xr

data = qndata.stocks.load_data()
print("type:", type(data))
print("dims:", data.dims)
print("coords:", list(data.coords))
if "field" in data.coords:
    print("fields:", list(data.coords["field"].values)[:20])

In [ ]:
import numpy as np
import xarray as xr
import qnt.data as qndata
import qnt.output as qnout


def _get_da_field(data: xr.DataArray, field_name: str) -> xr.DataArray:
    """
    Safely extract a (time, asset) DataArray for a given field from a DataArray that may
    or may not have a 'field' coord/dim.
    """
    # Case A: has 'field' -> select by field
    if ("field" in data.dims) or ("field" in data.coords):
        fields = list(data.coords["field"].values)
        if field_name not in fields:
            raise KeyError(f"Field '{field_name}' not found. Available fields: {fields}")
        return data.sel(field=field_name)

    # Case B: no 'field' -> assume data itself is the price series (likely close)
    if field_name == "close":
        return data
    raise KeyError(
        f"Your data has no 'field' dimension/coord, so '{field_name}' can't be extracted."
        f" Dims are: {data.dims}. Coords are: {list(data.coords)}"
    )


def generate_random_equal_weight_portfolio(seed=7, n_stocks=10):
    """
    Generates an equal-weight portfolio of n_stocks random assets.
    If 'is_liquid' exists, samples only liquid assets (last day). Otherwise samples from all assets.
    Returns: chosen_assets, weights(time, asset)
    """
    data = qndata.stocks.load_data()

    close = _get_da_field(data, "close")  # (time, asset)

    # Try to use liquidity if available
    chosen_universe = close.asset.values
    if ("field" in data.dims) or ("field" in data.coords):
        # only then can we try to pull is_liquid from fields
        try:
            is_liquid = _get_da_field(data, "is_liquid")
            liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
            chosen_universe = close.asset.values[liquid_last.values]
        except KeyError:
            pass  # no is_liquid field -> fall back to all assets

    if len(chosen_universe) < n_stocks:
        raise ValueError(f"Not enough assets ({len(chosen_universe)}) to pick {n_stocks}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(chosen_universe, size=n_stocks, replace=False)

    w = xr.DataArray(
        0.0,
        dims=("time", "asset"),
        coords={"time": close.time.values, "asset": close.asset.values},
        name="weights",
    )
    w.loc[dict(asset=chosen_assets)] = 1.0 / n_stocks  # 0.1 each if n_stocks=10

    # Clean is optional; keep it if you're using Quantiacs submission/backtest tooling
    try:
        w = qnout.clean(data, w)
    except Exception:
        # If clean fails due to format mismatch, portfolio weights are still valid for our mean/var calc
        pass

    return chosen_assets, w


def compute_portfolio_mean_variance(chosen_assets, annualize=252, use_log_returns=False):
    """
    Computes historical mean and variance of an equal-weight portfolio over chosen_assets
    using daily returns derived from close prices.
    """
    data = qndata.stocks.load_data()
    close = _get_da_field(data, "close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")
    R = rets.values  # (T, n)

    mu = np.nanmean(R, axis=0)          # asset daily means
    Sigma = np.cov(R, rowvar=False)     # daily covariance

    n = len(chosen_assets)
    w = np.ones(n) / n                  # 0.1 each if n=10

    mu_p = float(w @ mu)
    var_p = float(w @ Sigma @ w)

    return {
        "portfolio_mean_daily": mu_p,
        "portfolio_var_daily": var_p,
        "portfolio_mean_annual": mu_p * annualize,
        "portfolio_var_annual": var_p * annualize,
        "portfolio_vol_annual": float(np.sqrt(var_p * annualize)),
    }


# ---- Example run ----
chosen_assets, weights = generate_random_equal_weight_portfolio(seed=7, n_stocks=10)
stats = compute_portfolio_mean_variance(chosen_assets)
print("Chosen assets:", chosen_assets)
print(stats)

In [ ]:
import numpy as np

close = data.sel(field="close").sel(asset=chosen_assets)
rets = close / close.shift(time=1) - 1
rets = rets.dropna("time")

# constant equal weights = 0.1 each
w = np.ones(len(chosen_assets)) / len(chosen_assets)

# portfolio returns: rp(t) = sum_i w_i * r_i(t)
rp = (rets * w).sum("asset")   # xarray DataArray over time
print(rp.head())

In [ ]:
import matplotlib.pyplot as plt

equity = (1 + rp).cumprod("time")
equity.plot()
plt.title("Random 10-stock equal-weight portfolio equity curve")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 0) Load Quantiacs data + pick a 10-stock universe (fixed)
# ============================================================
def load_quantiacs_universe(seed=7, n_stocks=10):
    """
    Loads Quantiacs stocks data (DataArray with dims: field,time,asset),
    picks n_stocks liquid assets (liquid on the last day), and returns:
      data, chosen_assets
    """
    data = qndata.stocks.load_data()  # DataArray(field,time,asset)
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_stocks, replace=False)

    return data, chosen_assets


# ============================================================
# 1) Build mu_t and Sigma_t from Quantiacs prices
# ============================================================
def compute_mu_sigma_series(
    data,
    chosen_assets,
    lookback=60,
    annualize=252,
    use_log_returns=False
):
    """
    Builds time series of:
      - mu_series[t]   = estimated mean returns vector (n,)
      - Sigma_series[t]= estimated covariance matrix (n,n)
    from rolling window of returns of length `lookback`.
    
    Returns:
      times (np.array, length T)
      mu_series (T,n)
      Sigma_series (T,n,n)
      returns_matrix (T,n)  # daily returns aligned to times
    """
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")                 # DataArray(time, asset)
    times = rets.time.values
    R = rets.values                             # (T, n)

    T, n = R.shape
    if T <= lookback:
        raise ValueError(f"Not enough history: T={T}, lookback={lookback}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    # Rolling estimates
    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t+1, :]                # inclusive window ending at t
        # If window is too short early on, still compute with what we have
        mu_series[t] = np.nanmean(window, axis=0)
        Sigma_series[t] = np.cov(window, rowvar=False) if window.shape[0] > 1 else np.eye(n) * 1e-6

        # Optional annualization (comment out if you want daily scale)
        mu_series[t] *= annualize
        Sigma_series[t] *= annualize

    return times, mu_series, Sigma_series, R


# ============================================================
# 2) Your CVXPY pieces (same as you wrote)
# ============================================================
def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []
    n = w.shape[0]
    if fully_invested:
        cons.append(cp.sum(w) == 1)
    if long_only:
        cons.append(w >= 0)
    if w_min is not None:
        cons.append(w >= w_min * np.ones(n))
    if w_max is not None:
        cons.append(w <= w_max * np.ones(n))
    return cons

def solve_one_step(mu, Sigma, w_prev, gamma=1.0, lam=1e-2, w_max=0.25):
    n = len(mu)
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)  # turnover proxy

    obj = cp.Maximize(mu @ w - gamma * cp.quad_form(w, Sigma) - lam * TC)
    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max))
    prob.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    return w.value

def solve_two_step(mu, Sigma, w_prev, gamma=1.0, rho=10.0, lam=1e-2, w_max=0.25):
    n = len(mu)

    # Step 1: target (no trading cost)
    w_star = cp.Variable(n)
    obj1 = cp.Maximize(mu @ w_star - gamma * cp.quad_form(w_star, Sigma))
    prob1 = cp.Problem(obj1, add_constraints(w_star, long_only=True, w_max=w_max))
    prob1.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    w_star_val = w_star.value

    # Step 2: trade toward target with trading penalty
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)

    track = (rho / 2.0) * cp.sum_squares(w - w_star_val)
    obj2 = cp.Minimize(track + lam * TC)
    prob2 = cp.Problem(obj2, add_constraints(w, long_only=True, w_max=w_max))
    prob2.solve(solver=cp.OSQP, warm_start=True, verbose=False)

    return w_star_val, w.value

def run(mu_series, Sigma_series, method="one", gamma=5.0, lam=0.06, rho=25.0, w_max=0.10):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n
    W = np.zeros((T, n))
    W_star = np.full((T, n), np.nan)

    for t in range(T):
        mu_t = mu_series[t]
        Sigma_t = Sigma_series[t]

        if method == "one":
            w_t = solve_one_step(mu_t, Sigma_t, w_prev, gamma=gamma, lam=lam, w_max=w_max)
        else:
            w_star_t, w_t = solve_two_step(mu_t, Sigma_t, w_prev, gamma=gamma, rho=rho, lam=lam, w_max=w_max)
            W_star[t] = w_star_t

        W[t] = w_t
        w_prev = w_t

    return W, W_star

def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


# ============================================================
# 3) End-to-end: Quantiacs -> mu/Sigma -> one-step vs two-step
# ============================================================
# --- A) Build a universe of 10 random liquid stocks
data, chosen_assets = load_quantiacs_universe(seed=7, n_stocks=100)
print("Chosen assets:", chosen_assets)

# --- B) Build rolling mu_t and Sigma_t from Quantiacs prices
times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
    data,
    chosen_assets,
    lookback=60,        # change if you want
    annualize=252,      # keep consistent with your objective scaling
    use_log_returns=False
)

# --- C) Run optimizations
W_one, _ = run(mu_series, Sigma_series, method="one", gamma=5.0, lam=0.06, w_max=0.1)
W_two, W_star = run(mu_series, Sigma_series, method="two", gamma=5.0, lam=0.06, rho=25.0, w_max=0.1)

to_one = turnover(W_one)
to_two = turnover(W_two)

# ============================================================
# 4) Plots (same as your synthetic example)
# ============================================================
# Plot A: turnover
plt.figure()
plt.plot(to_one, label="One-step turnover")
plt.plot(to_two, label="Two-step turnover")
plt.title("Turnover per rebalance: sum |Δw|")
plt.xlabel("t")
plt.ylabel("sum |Δw|")
plt.legend()
plt.tight_layout()
plt.show()

# Plot B: weight heatmaps
plt.figure()
plt.imshow(W_one.T, aspect="auto")
plt.title("One-step weights heatmap (assets x time)")
plt.xlabel("t")
plt.ylabel("asset")
plt.colorbar(label="weight")
plt.tight_layout()
plt.show()

plt.figure()
plt.imshow(W_two.T, aspect="auto")
plt.title("Two-step weights heatmap (assets x time)")
plt.xlabel("t")
plt.ylabel("asset")
plt.colorbar(label="weight")
plt.tight_layout()
plt.show()

# Plot C: a few assets over time
k = min(5, len(chosen_assets))
plt.figure()
for i in range(k):
    plt.plot(W_one[:, i], label=f"{chosen_assets[i]}")
plt.title("One-step: sample asset weights")
plt.xlabel("t")
plt.ylabel("weight")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure()
for i in range(k):
    plt.plot(W_two[:, i], label=f"{chosen_assets[i]}")
plt.title("Two-step: sample asset weights")
plt.xlabel("t")
plt.ylabel("weight")
plt.legend()
plt.tight_layout()
plt.show()

# Plot D: tracking distance in two-step
dist = np.linalg.norm(W_two - W_star, axis=1)
plt.figure()
plt.plot(dist)
plt.title("Two-step tracking distance ||w_t - w*_t||_2")
plt.xlabel("t")
plt.ylabel("L2 distance")
plt.tight_layout()
plt.show()


# ============================================================
# 5) (Optional but useful) Realized portfolio return/equity curves
# ============================================================
# Use DAILY returns (not annualized) for realized equity curve
# R_daily is daily returns matrix aligned to times
rp_one = np.sum(R_daily * W_one, axis=1)  # daily portfolio returns
rp_two = np.sum(R_daily * W_two, axis=1)

eq_one = np.cumprod(1 + rp_one)
eq_two = np.cumprod(1 + rp_two)

plt.figure()
plt.plot(eq_one, label="One-step equity")
plt.plot(eq_two, label="Two-step equity")
plt.title("Realized equity curves (using Quantiacs daily returns)")
plt.xlabel("t")
plt.ylabel("equity")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.plot(eq_one, label="One-step equity")
plt.title("Realized equity curves (using Quantiacs daily returns)")
plt.xlabel("t")
plt.ylabel("equity")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.plot(eq_two, label="Two-step equity")
plt.title("Realized equity curves (using Quantiacs daily returns)")
plt.xlabel("t")
plt.ylabel("equity")
plt.legend()
plt.tight_layout()
plt.show()

I annualized mu_series and Sigma_series inside compute_mu_sigma_series so your objective scale is stable. If you prefer everything in daily units, set annualize=None and remove the multiplications.

lookback=60 means your 
𝜇
𝑡
,
Σ
𝑡
μ
t
	​

,Σ
t
	​

 are estimated from the last ~60 trading days. You can test sensitivity: 20/60/120.

The “Data loaded / forward filling” messages are normal in Quantiacs.

What the equity curves suggest

Two-step (orange) outperforms one-step (blue) on realized equity

In your run, the two-step ends much higher (roughly ~2× the ending equity of one-step).

That means the combination of (i) target portfolio from mean–variance and (ii) tracking/trading step produced better realized returns on this sample.

Two-step likely trades “more smoothly” into positions

Even if both methods use an L1 turnover penalty, the two-step has an extra mechanism: it trades toward a target 
𝑤
𝑡
\*
w
t
\*
	​

 with a quadratic tracking term.

That often produces more stable transitions (less “jerky” day-to-day changes), and can reduce the damage from estimation noise in 
𝜇
𝑡
μ
t
	​

.

But: your equity plot alone doesn’t prove turnover is lower — you should check the turnover plot (sum |Δw|) you generated earlier.
Two-step and one-step heatmaps look broadly similar

This is interesting: it suggests that the “target” portfolios implied by the two approaches aren’t wildly different.

Yet the realized equity differs a lot — meaning the benefit might come from how the two-step trades toward the target (smoother, less overreacting), not necessarily from a completely different asset selection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + pick random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()  # DataArray(field, time, asset)
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Rolling mu_t and Sigma_t (1y lookback) + annualization
# ============================================================
def compute_mu_sigma_series(data, chosen_assets, lookback=252, annualize=252, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")  # DataArray(time, asset)
    times = rets.time.values
    R = rets.values  # (T, n)

    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback+1}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t+1, :]  # inclusive end at t

        mu = np.nanmean(window, axis=0)
        # sample covariance; add tiny jitter if window too short
        if window.shape[0] > 2:
            Sigma = np.cov(window, rowvar=False)
        else:
            Sigma = np.eye(n) * 1e-6

        # annualize to keep objective scaling stable
        mu_series[t] = mu * annualize
        Sigma_series[t] = Sigma * annualize

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) PSD enforcement (fixes CVXPY DCPError)
# ============================================================
def make_psd(Sigma, ridge=1e-3, eps=1e-8):
    """
    Symmetrize + ridge regularization, then wrap as PSD for CVXPY.
    This avoids DCPError when sample covariance is numerically indefinite.
    """
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return cp.psd_wrap(cp.Constant(S))


# ============================================================
# 4) CVXPY optimization: one-step and two-step
# ============================================================
def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []
    n = w.shape[0]
    if fully_invested:
        cons.append(cp.sum(w) == 1)
    if long_only:
        cons.append(w >= 0)
    if w_min is not None:
        cons.append(w >= w_min * np.ones(n))
    if w_max is not None:
        cons.append(w <= w_max * np.ones(n))
    return cons


def solve_one_step(mu, Sigma, w_prev, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)

    Sigma_psd = make_psd(Sigma, ridge=ridge)

    obj = cp.Maximize(mu @ w - gamma * cp.quad_form(w, Sigma_psd) - lam * TC)
    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max))
    prob.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    return w.value


def solve_two_step(mu, Sigma, w_prev, gamma=5.0, rho=25.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    Sigma_psd = make_psd(Sigma, ridge=ridge)

    # Step 1: target portfolio (no trading cost)
    w_star = cp.Variable(n)
    obj1 = cp.Maximize(mu @ w_star - gamma * cp.quad_form(w_star, Sigma_psd))
    prob1 = cp.Problem(obj1, add_constraints(w_star, long_only=True, w_max=w_max))
    prob1.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    w_star_val = w_star.value

    # Step 2: trade toward target with turnover + tracking penalty
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)
    track = (rho / 2.0) * cp.sum_squares(w - w_star_val)

    obj2 = cp.Minimize(track + lam * TC)
    prob2 = cp.Problem(obj2, add_constraints(w, long_only=True, w_max=w_max))
    prob2.solve(solver=cp.OSQP, warm_start=True, verbose=False)

    return w_star_val, w.value


def run(mu_series, Sigma_series, method="one", gamma=5.0, lam=0.06, rho=25.0, w_max=0.10, ridge=1e-3):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n

    W = np.zeros((T, n))
    W_star = np.full((T, n), np.nan)

    for t in range(T):
        mu_t = mu_series[t]
        Sigma_t = Sigma_series[t]

        if method == "one":
            w_t = solve_one_step(mu_t, Sigma_t, w_prev, gamma=gamma, lam=lam, w_max=w_max, ridge=ridge)
        else:
            w_star_t, w_t = solve_two_step(mu_t, Sigma_t, w_prev, gamma=gamma, rho=rho, lam=lam, w_max=w_max, ridge=ridge)
            W_star[t] = w_star_t

        W[t] = w_t
        w_prev = w_t

    return W, W_star


def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


# ============================================================
# 5) End-to-end run: 50 assets, 1y lookback
# ============================================================
seed = 7
n_assets = 50
lookback = 252

gamma = 5.0
lam = 0.06
rho = 25.0
w_max = 0.10          # max 10% per asset (feasible since n_assets=50)
ridge = 1e-3          # PSD regularization strength

data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
print("Chosen assets (first 10):", chosen_assets[:10], " ... total:", len(chosen_assets))

times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
    data, chosen_assets, lookback=lookback, annualize=252, use_log_returns=False
)

W_one, _ = run(mu_series, Sigma_series, method="one", gamma=gamma, lam=lam, rho=rho, w_max=w_max, ridge=ridge)
W_two, W_star = run(mu_series, Sigma_series, method="two", gamma=gamma, lam=lam, rho=rho, w_max=w_max, ridge=ridge)

to_one = turnover(W_one)
to_two = turnover(W_two)

# ============================================================
# 6) Plots: turnover + heatmaps + equity (causal shift)
# ============================================================
# A) Turnover
plt.figure()
plt.plot(to_one, label="One-step turnover")
plt.plot(to_two, label="Two-step turnover")
plt.title("Turnover per rebalance: sum |Δw|")
plt.xlabel("t")
plt.ylabel("sum |Δw|")
plt.legend()
plt.tight_layout()
plt.show()

# B) Weight heatmaps
plt.figure()
plt.imshow(W_one.T, aspect="auto")
plt.title("One-step weights heatmap (assets x time)")
plt.xlabel("t")
plt.ylabel("asset")
plt.colorbar(label="weight")
plt.tight_layout()
plt.show()

plt.figure()
plt.imshow(W_two.T, aspect="auto")
plt.title("Two-step weights heatmap (assets x time)")
plt.xlabel("t")
plt.ylabel("asset")
plt.colorbar(label="weight")
plt.tight_layout()
plt.show()

# C) A few assets over time
k = 5
plt.figure()
for i in range(k):
    plt.plot(W_one[:, i], label=f"{chosen_assets[i]}")
plt.title("One-step: sample asset weights")
plt.xlabel("t")
plt.ylabel("weight")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure()
for i in range(k):
    plt.plot(W_two[:, i], label=f"{chosen_assets[i]}")
plt.title("Two-step: sample asset weights")
plt.xlabel("t")
plt.ylabel("weight")
plt.legend()
plt.tight_layout()
plt.show()

# D) Tracking distance
dist = np.linalg.norm(W_two - W_star, axis=1)
plt.figure()
plt.plot(dist)
plt.title("Two-step tracking distance ||w_t - w*_t||_2")
plt.xlabel("t")
plt.ylabel("L2 distance")
plt.tight_layout()
plt.show()

# E) Realized equity curves (IMPORTANT: shift weights by 1 day to avoid look-ahead)
# R_daily[t] is return from t-1 -> t. Use weights from t-1.
rp_one = np.sum(R_daily[1:] * W_one[:-1], axis=1)
rp_two = np.sum(R_daily[1:] * W_two[:-1], axis=1)

eq_one = np.cumprod(1 + rp_one)
eq_two = np.cumprod(1 + rp_two)

plt.figure()
plt.plot(eq_one, label="One-step equity")
plt.plot(eq_two, label="Two-step equity")
plt.title("Realized equity curves (weights shifted by 1 day)")
plt.xlabel("t")
plt.ylabel("equity")
plt.legend()
plt.tight_layout()
plt.show()

different lookback and frequency 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + pick random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()  # DataArray(field, time, asset)
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Rolling mu_t and Sigma_t with separate lookbacks
# ============================================================
def compute_mu_sigma_series(
    data,
    chosen_assets,
    mu_lookback=63,
    sigma_lookback=504,
    annualize=252,
    use_log_returns=False
):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")  # DataArray(time, asset)
    times = rets.time.values
    R = rets.values  # (T, n)

    T, n = R.shape
    min_needed = max(mu_lookback, sigma_lookback) + 1
    if T <= min_needed:
        raise ValueError(f"Not enough history: T={T}, need > {min_needed}.")

    mu_series = np.zeros((T, n))
    Sigma_series = np.zeros((T, n, n))

    for t in range(T):
        mu_start = max(0, t - mu_lookback)
        sigma_start = max(0, t - sigma_lookback)

        mu_window = R[mu_start:t+1, :]
        sigma_window = R[sigma_start:t+1, :]

        mu = np.nanmean(mu_window, axis=0)

        if sigma_window.shape[0] > 2:
            Sigma = np.cov(sigma_window, rowvar=False)
        else:
            Sigma = np.eye(n) * 1e-6

        mu_series[t] = mu * annualize
        Sigma_series[t] = Sigma * annualize

    return times, mu_series, Sigma_series, R


# ============================================================
# 3) PSD enforcement
# ============================================================
def make_psd(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return cp.psd_wrap(cp.Constant(S))


# ============================================================
# 4) CVXPY optimization: one-step and two-step
# ============================================================
def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []
    n = w.shape[0]
    if fully_invested:
        cons.append(cp.sum(w) == 1)
    if long_only:
        cons.append(w >= 0)
    if w_min is not None:
        cons.append(w >= w_min * np.ones(n))
    if w_max is not None:
        cons.append(w <= w_max * np.ones(n))
    return cons


def solve_one_step(mu, Sigma, w_prev, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)

    Sigma_psd = make_psd(Sigma, ridge=ridge)

    obj = cp.Maximize(mu @ w - gamma * cp.quad_form(w, Sigma_psd) - lam * TC)
    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max))
    prob.solve(solver=cp.OSQP, warm_start=True, verbose=False)

    if w.value is None:
        return w_prev.copy()
    return w.value


def solve_two_step(mu, Sigma, w_prev, gamma=5.0, rho=25.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    Sigma_psd = make_psd(Sigma, ridge=ridge)

    # Step 1: target portfolio (no trading cost)
    w_star = cp.Variable(n)
    obj1 = cp.Maximize(mu @ w_star - gamma * cp.quad_form(w_star, Sigma_psd))
    prob1 = cp.Problem(obj1, add_constraints(w_star, long_only=True, w_max=w_max))
    prob1.solve(solver=cp.OSQP, warm_start=True, verbose=False)

    if w_star.value is None:
        return w_prev.copy(), w_prev.copy()

    w_star_val = w_star.value

    # Step 2: trade toward target with turnover + tracking penalty
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)
    track = (rho / 2.0) * cp.sum_squares(w - w_star_val)

    obj2 = cp.Minimize(track + lam * TC)
    prob2 = cp.Problem(obj2, add_constraints(w, long_only=True, w_max=w_max))
    prob2.solve(solver=cp.OSQP, warm_start=True, verbose=False)

    if w.value is None:
        return w_star_val, w_prev.copy()

    return w_star_val, w.value


# ============================================================
# 5) Rebalance scheduler
# ============================================================
def should_rebalance(t, rebalance_every=1, start_at=0):
    return (t - start_at) % rebalance_every == 0


def run(
    mu_series,
    Sigma_series,
    method="one",
    gamma=5.0,
    lam=0.06,
    rho=25.0,
    w_max=0.10,
    ridge=1e-3,
    rebalance_every=1
):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n

    W = np.zeros((T, n))
    W_star = np.full((T, n), np.nan)

    for t in range(T):
        mu_t = mu_series[t]
        Sigma_t = Sigma_series[t]

        if should_rebalance(t, rebalance_every=rebalance_every):
            if method == "one":
                w_t = solve_one_step(
                    mu_t, Sigma_t, w_prev,
                    gamma=gamma, lam=lam, w_max=w_max, ridge=ridge
                )
            else:
                w_star_t, w_t = solve_two_step(
                    mu_t, Sigma_t, w_prev,
                    gamma=gamma, rho=rho, lam=lam, w_max=w_max, ridge=ridge
                )
                W_star[t] = w_star_t
        else:
            w_t = w_prev.copy()
            if method == "two":
                W_star[t] = np.nan

        W[t] = w_t
        w_prev = w_t

    return W, W_star


def turnover(W):
    return np.sum(np.abs(np.diff(W, axis=0)), axis=1)


def realized_portfolio_returns(R_daily, W):
    # R_daily[t] is return from t-1 -> t, so use weights from t-1
    return np.sum(R_daily[1:] * W[:-1], axis=1)


def annualized_sharpe(rp, annualize=252):
    rp = np.asarray(rp)
    vol = np.std(rp)
    if vol < 1e-12:
        return np.nan
    return np.sqrt(annualize) * np.mean(rp) / vol


# ============================================================
# 6) End-to-end run
# ============================================================
seed = 7
n_assets = 50

# Separate lookbacks
mu_lookback = 63        # shorter lookback for expected returns
sigma_lookback = 504    # longer lookback for covariance

# Rebalance frequency:
# 1 = daily, 5 = weekly, 10 = biweekly, 21 = monthly
rebalance_every = 5

gamma = 5.0
lam = 0.10
rho = 25.0
w_max = 0.10
ridge = 1e-2

data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
print("Chosen assets (first 10):", chosen_assets[:10], " ... total:", len(chosen_assets))

times, mu_series, Sigma_series, R_daily = compute_mu_sigma_series(
    data,
    chosen_assets,
    mu_lookback=mu_lookback,
    sigma_lookback=sigma_lookback,
    annualize=252,
    use_log_returns=False
)

W_one, _ = run(
    mu_series, Sigma_series,
    method="one",
    gamma=gamma, lam=lam, rho=rho,
    w_max=w_max, ridge=ridge,
    rebalance_every=rebalance_every
)

W_two, W_star = run(
    mu_series, Sigma_series,
    method="two",
    gamma=gamma, lam=lam, rho=rho,
    w_max=w_max, ridge=ridge,
    rebalance_every=rebalance_every
)

to_one = turnover(W_one)
to_two = turnover(W_two)

rp_one = realized_portfolio_returns(R_daily, W_one)
rp_two = realized_portfolio_returns(R_daily, W_two)

eq_one = np.cumprod(1 + rp_one)
eq_two = np.cumprod(1 + rp_two)

print(f"One-step annualized Sharpe: {annualized_sharpe(rp_one):.3f}")
print(f"Two-step annualized Sharpe: {annualized_sharpe(rp_two):.3f}")
print(f"Average one-step turnover: {np.mean(to_one):.4f}")
print(f"Average two-step turnover: {np.mean(to_two):.4f}")


# ============================================================
# 7) Plots
# ============================================================
# A) Turnover
plt.figure()
plt.plot(to_one, label="One-step turnover")
plt.plot(to_two, label="Two-step turnover")
plt.title(f"Turnover per day | rebalance_every={rebalance_every}")
plt.xlabel("t")
plt.ylabel("sum |Δw|")
plt.legend()
plt.tight_layout()
plt.show()

# B) Weight heatmaps
plt.figure()
plt.imshow(W_one.T, aspect="auto")
plt.title("One-step weights heatmap (assets x time)")
plt.xlabel("t")
plt.ylabel("asset")
plt.colorbar(label="weight")
plt.tight_layout()
plt.show()

plt.figure()
plt.imshow(W_two.T, aspect="auto")
plt.title("Two-step weights heatmap (assets x time)")
plt.xlabel("t")
plt.ylabel("asset")
plt.colorbar(label="weight")
plt.tight_layout()
plt.show()

# C) A few assets over time
k = 5
plt.figure()
for i in range(k):
    plt.plot(W_one[:, i], label=f"{chosen_assets[i]}")
plt.title("One-step: sample asset weights")
plt.xlabel("t")
plt.ylabel("weight")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure()
for i in range(k):
    plt.plot(W_two[:, i], label=f"{chosen_assets[i]}")
plt.title("Two-step: sample asset weights")
plt.xlabel("t")
plt.ylabel("weight")
plt.legend()
plt.tight_layout()
plt.show()

# D) Tracking distance (only meaningful on rebalance dates)
dist = np.linalg.norm(np.nan_to_num(W_two - W_star, nan=0.0), axis=1)
plt.figure()
plt.plot(dist)
plt.title("Two-step tracking distance")
plt.xlabel("t")
plt.ylabel("L2 distance")
plt.tight_layout()
plt.show()

# E) Equity curves
plt.figure()
plt.plot(eq_one, label="One-step equity")
plt.plot(eq_two, label="Two-step equity")
plt.title("Realized equity curves (weights shifted by 1 day)")
plt.xlabel("t")
plt.ylabel("equity")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL: Performance summary for current code
# ============================================================
import numpy as np
import pandas as pd

def max_drawdown(equity_curve):
    equity_curve = pd.Series(equity_curve).dropna()
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1.0
    return drawdown.min()

def annualized_sharpe(port_rets, periods_per_year=252):
    port_rets = pd.Series(port_rets).dropna()
    mu = port_rets.mean()
    sigma = port_rets.std()
    if sigma == 0 or np.isnan(sigma):
        return np.nan
    return np.sqrt(periods_per_year) * mu / sigma

def average_turnover_per_rebalance(turnover_series, tol=1e-12):
    turnover_series = pd.Series(turnover_series).dropna()
    rebalance_turnover = turnover_series[turnover_series > tol]
    if len(rebalance_turnover) == 0:
        return 0.0
    return rebalance_turnover.mean()

summary = pd.DataFrame({
    "One-Step": {
        "Final Equity": eq_one[-1],
        "Sharpe": annualized_sharpe(rp_one),
        "Max Drawdown": max_drawdown(eq_one),
        "Average Daily Turnover": np.mean(to_one),
        "Average Turnover per Rebalance": average_turnover_per_rebalance(to_one),
        "mu_lookback": mu_lookback,
        "sigma_lookback": sigma_lookback,
        "rebalance_every": rebalance_every,
    },
    "Two-Step": {
        "Final Equity": eq_two[-1],
        "Sharpe": annualized_sharpe(rp_two),
        "Max Drawdown": max_drawdown(eq_two),
        "Average Daily Turnover": np.mean(to_two),
        "Average Turnover per Rebalance": average_turnover_per_rebalance(to_two),
        "mu_lookback": mu_lookback,
        "sigma_lookback": sigma_lookback,
        "rebalance_every": rebalance_every,
    }
}).T

summary

In [ ]:
def plot_rebalance_snapshot(W_two, W_star, t, chosen_assets=None, top_k=15):
    w_prev = W_two[t-1]
    w_exec = W_two[t]
    w_target = W_star[t]

    if np.all(np.isnan(w_target)):
        print(f"No target portfolio stored at t={t}. Choose a rebalance date.")
        return

    # focus on the largest target weights
    idx = np.argsort(-w_target)[:top_k]

    labels = [str(i) for i in idx] if chosen_assets is None else [str(chosen_assets[i]) for i in idx]

    x = np.arange(len(idx))
    width = 0.25

    plt.figure(figsize=(12, 5))
    plt.bar(x - width, w_prev[idx], width=width, label="w_prev")
    plt.bar(x,         w_target[idx], width=width, label="w_star")
    plt.bar(x + width, w_exec[idx], width=width, label="w_exec")

    plt.xticks(x, labels, rotation=45)
    plt.ylabel("weight")
    plt.title(f"Rebalance snapshot at t={t}")
    plt.legend()
    plt.tight_layout()
    plt.show()
t = 100
plot_rebalance_snapshot(W_two, W_star, t, chosen_assets=chosen_assets, top_k=20)

In [ ]:
def plot_trade_vector(W_two, W_star, t, chosen_assets=None, top_k=15):
    w_prev = W_two[t-1]
    w_exec = W_two[t]
    trade = w_exec - w_prev

    idx = np.argsort(-np.abs(trade))[:top_k]
    labels = [str(i) for i in idx] if chosen_assets is None else [str(chosen_assets[i]) for i in idx]

    plt.figure(figsize=(12, 5))
    plt.bar(np.arange(len(idx)), trade[idx])
    plt.xticks(np.arange(len(idx)), labels, rotation=45)
    plt.axhline(0, linewidth=1)
    plt.ylabel("trade = w_t - w_{t-1}")
    plt.title(f"Largest trades at t={t}")
    plt.tight_layout()
    plt.show()
plot_trade_vector(W_two, W_star, t=100, chosen_assets=chosen_assets, top_k=15)